In [ ]:
import numpy as np
from scipy import signal
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pyedflib
import mne
import pywt
import os
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='edf_processing.log'
)

def morlet_wavelet(f0, t, n_cycles):
    sigma = n_cycles / (2 * np.pi * f0)
    norm_factor = (np.pi * sigma**2) ** (-0.25)
    return norm_factor * np.exp(2j * np.pi * f0 * t) * np.exp(-t**2 / (2 * sigma**2))

def process_frequency(args):
    signal_data, freq, samplerate, order_min, order_max = args
    n_orders = order_max - order_min + 1
    transform_product = np.ones_like(signal_data, dtype=np.float32)
    
    pad_length = int(6 * samplerate / freq)  # 6 cycles padding
    padded_signal = np.pad(signal_data, pad_length, mode='reflect')
    
    for order in range(order_min, order_max + 1):
        try:
            t_wavelet = np.arange(-3 * order / (2 * np.pi * freq), 
                                  3 * order / (2 * np.pi * freq), 
                                  1 / samplerate)
            wavelet = morlet_wavelet(freq, t_wavelet, order)
            conv = signal.convolve(padded_signal, wavelet, mode='same')
            start_idx = (len(conv) - len(signal_data)) // 2
            conv = conv[start_idx:start_idx + len(signal_data)]  
            transform_product *= np.abs(conv)**(1/n_orders)
        except Exception as e:
            logging.error(f"Error at {freq}Hz/order{order}: {str(e)}")
            continue
    
    return transform_product

def superlet_transform(signal_data, samplerate, freq_range, order_min=3, order_max=15, n_jobs=4):
    signal_data = signal_data.astype(np.float32)
    args = [(signal_data, f, samplerate, order_min, order_max) for f in freq_range]
    
    with ThreadPoolExecutor(max_workers=n_jobs) as executor:
        results = list(tqdm(executor.map(process_frequency, args), total=len(freq_range)))
    
    transform = np.stack(results)
    return transform / transform.max(), np.arange(len(signal_data)) / samplerate

def wavelet_denoising(signal_data, wavelet='db4', level=4):
    coeffs = pywt.wavedec(signal_data, wavelet, level=level)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    uthresh = sigma * np.sqrt(2 * np.log(len(signal_data)))
    coeffs_thresholded = [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs]
    return pywt.waverec(coeffs_thresholded, wavelet)[:len(signal_data)]

def plot_superlet_overlay(transforms, times, freq_range, save_path=None):
    plt.figure(figsize=(12, 6), dpi=300)
    for transform in transforms:
        plt.imshow(np.abs(transform),
                   aspect='auto',
                   origin='lower',
                   extent=[times[0], times[-1], freq_range[0], freq_range[-1]],
                   cmap='jet',
                   norm=LogNorm(vmin=1e-4, vmax=1),  # Adjusted for better clarity
                   alpha=0.2)  # Adjust transparency to match first image
    plt.axis('off')  # Remove axis
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.show()

def validate_edf_file(file_path):
    """Validate if file is a proper EDF file"""
    try:
        with pyedflib.EdfReader(file_path) as f:
            return True
    except OSError:
        logging.error(f"Invalid EDF file: {file_path}")
        return False
    except Exception as e:
        logging.error(f"Error validating {file_path}: {str(e)}")
        return False

def read_edf_file(file_path):
    """Read EDF file with error handling"""
    try:
        with pyedflib.EdfReader(file_path) as f:
            n = f.signals_in_file
            signal_data = np.array([f.readSignal(i) for i in range(n)])
            samplerate = f.getSampleFrequency(0)
            return signal_data, samplerate
    except Exception as e:
        logging.error(f"Error reading {file_path}: {str(e)}")
        return None, None

def apply_ica(signal_data, samplerate):
    info = mne.create_info(ch_names=[f'CH{i}' for i in range(signal_data.shape[0])],
                           sfreq=samplerate, ch_types='eeg')
    raw = mne.io.RawArray(signal_data, info)
    raw.filter(1.0, None)  # High-pass filter with a lower bound of 1 Hz
    ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter=800, method='fastica')
    ica.fit(raw)
    raw_cleaned = ica.apply(raw)
    return raw_cleaned.get_data()

def process_folder(input_folder, output_folder, label):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for file_name in os.listdir(input_folder):
        if file_name.endswith('.edf'):
            file_path = os.path.join(input_folder, file_name)
            
            # Validate file first
            if not validate_edf_file(file_path):
                continue
                
            # Read file
            signal_data, samplerate = read_edf_file(file_path)
            if signal_data is None:
                continue
            
            try:
                # Apply ICA
                clean_data = apply_ica(signal_data, samplerate)
                
                # Apply wavelet denoising
                denoised_data = np.array([wavelet_denoising(ch) for ch in clean_data])
                
                freq_range = np.arange(1, 40, 1)  # Adjusted upper frequency to match first image
                all_transforms = []
                for channel_data in denoised_data:
                    transform, times = superlet_transform(channel_data, samplerate, freq_range)
                    all_transforms.append(transform)
                
                # Plot the transforms by overlaying
                output_path = os.path.join(output_folder, f'{label}_{file_name[:-4]}.png')
                plot_superlet_overlay(all_transforms, times, freq_range, save_path=output_path)
                logging.info(f"Successfully processed: {file_path}")
                
            except Exception as e:
                logging.error(f"Error processing {file_path}: {str(e)}")

def main():
    healthy_folder = r"D:\Contourlet-CNN\Code\DS\Healthy"
    mdd_folder = r"D:\Contourlet-CNN\Code\DS\MDD"
    output_folder = r"D:\Contourlet-CNN\Code\DS\Balance"
    
    process_folder(healthy_folder, os.path.join(output_folder, 'Healthy'), 'healthy')
    process_folder(mdd_folder, os.path.join(output_folder, 'MDD'), 'mdd')

if __name__ == "__main__":
    main()